## Kedro Setup

In [ ]:
import os

os.environ["PYSPARK_PIN_THREAD"] = "false"


In [ ]:
%load_ext kedro.ipython

## Time Series Cross Validation

In [ ]:
# calendar = catalog.load("calendar")

# SATURDAY_ID = 7
# calendar_processed = (
#     calendar.withColumn("year", F.year("date"))
#     .withColumn("week", F.weekofyear("date"))
#     .filter(F.dayofweek("date") == SATURDAY_ID)
#     .withColumn("time_percentage", F.percent_rank().over(W.orderBy("date")))
#     .withColumn("row_num", F.row_number().over(W.orderBy("date")))
#     .sort(F.asc("date"))
#     .toPandas()
# )
# num_dates = calendar_processed.shape[0]

# outer_tscv = TimeSeriesSplit(n_splits=3)
# for outer_fold, (train_idx, val_idx) in enumerate(outer_tscv.split(calendar_processed)):
#     # Generate train, val separation indices
#     min_train, max_train = 0, max(train_idx)
#     min_val, max_val = max_train + 1, max(val_idx)

#     # Time percentage indicators
#     min_train_perc, max_train_perc = min_train, max_train / num_dates
#     min_val_perc, max_val_perc = min_val / num_dates, max_val / num_dates

#     print(f"Outer TSCV {outer_fold + 1}")
#     print(50*"-")
#     print(f"Train Span: {min_train_perc, max_train_perc}")
#     print(f"Val Span: {min_val_perc, max_val_perc}")
#     print(50*"=")


#     inner_tscv = TimeSeriesSplit(n_splits=3)
#     for inner_fold, (cv_train_idx, cv_val_idx) in enumerate(inner_tscv.split(train_idx)):

#         # Generate train, val separation indices
#         min_train_inner, max_train_inner = 0, max(cv_train_idx)
#         min_val_inner, max_val_inner = max_train_inner + 1, max(cv_val_idx)

#         # Time percentage indicators
#         min_train_perc_inner, max_train_perc_inner = min_train_inner, max_train_inner / num_dates
#         min_val_perc_inner, max_val_perc_inner = min_val_inner / num_dates, max_val_inner / num_dates
#         print(f"Inner TSCV {inner_fold + 1}")
#         print(f"Train Span: {min_train_perc_inner, max_train_perc_inner}")
#         print(f"Val Span: {min_val_perc_inner, max_val_perc_inner}")

#     print("\n")


## Base Model Training Prototyping

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, List, Dict, Any


import pyspark.sql.functions as F
from pyspark.sql.types import FloatType
from pyspark.sql import Window, WindowSpec, DataFrame, Column
# from pyspark.ml.pipeline import Pipeline, PipelineModel
# from pyspark.ml.feature import (
#     StringIndexer,
#     VectorAssembler,
#     SQLTransformer,
#     QuantileDiscretizer,
#     PolynomialExpansion,
#     StandardScaler,
# )
# from pyspark.ml.regression import (
#     DecisionTreeRegressor,
#     RandomForestRegressor,
#     LinearRegression,
#     GBTRegressor,
# )
# from pyspark.ml.stat import Correlation
# from xgboost.spark import SparkXGBRegressor
# from pyspark.ml.evaluation import (
#     RegressionEvaluator,
#     BinaryClassificationEvaluator,
#     MulticlassClassificationEvaluator,
# )

In [ ]:
df = catalog.load("weekly_sales_filled")
calendar = catalog.load("calendar").withColumn(
    "percent_rank", F.percent_rank().over(Window.orderBy("date"))
)

# Suppose we do a train test split with 80, 15, 5
train_perc = 0.8
val_perc = 0.15
test_perc = 1 - train_perc + val_perc
cutoffs = (
    calendar.withColumn(
        "idx",
        F.when(F.col("percent_rank") < train_perc, "train")
        .when(F.col("percent_rank") < train_perc + val_perc, "val")
        .otherwise("test"),
    )
    .groupby("idx")
    .agg(F.max("date").alias("cutoff"))
    .toPandas()
    .set_index("idx")["cutoff"]
    .to_dict()
)

## Feature Engineering

In [ ]:
@dataclass
class SparkAggFn:
    prefix: str
    spark_function: Callable[[str, ...], Column]
    params: Dict[str, Any] = field(default_factory=dict)

In [ ]:
def add_lags(
    df: DataFrame, lag_col: str, wspec: WindowSpec, lags: list[int]
) -> DataFrame:
    lag_cols = {
        f"lag_{lag_col}_{i}": F.lag(col=lag_col, offset=i).over(wspec).cast(FloatType())
        for i in lags
    }

    return df.withColumns(lag_cols)


def add_agg_over_windows(
    df: DataFrame,
    val_col: str,
    spark_agg_fn: SparkAggFn,
    wspec: WindowSpec,
    periods: list[int],
) -> DataFrame:
    mean_cols = {
        f"{spark_agg_fn.prefix}_{val_col}_{i}": spark_agg_fn.spark_function(
            val_col, **spark_agg_fn.params
        ).over(wspec.rowsBetween(-i, -1)).cast(FloatType())
        for i in periods
    }

    return df.withColumns(mean_cols)

#### Resampled Features

In [ ]:
def define_grouping_alias(grouping_cols: List[str]) -> str:
    return f"{"_".join(grouping_cols)}_sale".replace("_id", "")


def add_product_rank(
    df: DataFrame, rolled_sales_col: str, output_col: str = "product_rank"
) -> DataFrame:
    return df.withColumn(
        output_col,
        F.percent_rank().over(
            Window.partitionBy("store_id", "date").orderBy(F.asc(rolled_sales_col))
        ),
    )


def resample_per_group(
    df: DataFrame,
    grouping_cols: List[str],
    grouping_alias: str,
    date_col: str = "date",
    label_col: str = "unit_sale",
) -> DataFrame:
    if not grouping_alias:
        grouping_alias = define_grouping_alias(grouping_cols)

    return df.groupby(*grouping_cols, date_col).agg(
        F.sum(label_col).alias(define_grouping_alias(grouping_cols))
    )

In [ ]:
def add_features_per_grouping(
    df: DataFrame,
    grouping_cols: List[str],
    lags: List[int],
    group_periods: List[int],
    agg_spark_fns: List[SparkAggFn],
) -> DataFrame:
    grouping_alias = define_grouping_alias(grouping_cols)
    group_wspec = Window.partitionBy(*grouping_cols).orderBy("date")

    group_week_sales = df.transform(
        resample_per_group,
        grouping_cols=grouping_cols,
        grouping_alias=grouping_alias,
    ).transform(
        add_lags,
        lag_col=grouping_alias,
        wspec=group_wspec,
        lags=lags,
    )

    for agg_fn in agg_spark_fns:
        group_week_sales = group_week_sales.transform(
            add_agg_over_windows,
            val_col=grouping_alias,
            spark_agg_fn=agg_fn,
            wspec=group_wspec,
            periods=group_periods,
        )

    return df.join(
        other=group_week_sales, on=grouping_cols + ["date"], how="left"
    ).drop(grouping_alias)


#### Temporal Encodings

In [ ]:
def add_smooth_rbf(
    df: DataFrame,
    peak_at: int,
    alpha: float,
    date_col: str = "date",
    period_length: int = 52,
    start_date: str = "2013-01-01",
) -> DataFrame:
    t = F.ceil(F.date_diff(F.col(date_col), F.to_date(F.lit(start_date))) / 7)
    smooth_rbf = F.exp(
        -(F.lit(period_length) ** 2)
        / F.lit(alpha)
        * F.sin(F.pi() * (t - F.lit(peak_at)) / F.lit(period_length)) ** 2
    )

    return df.withColumn(f"smooth_rbf_{peak_at}_{int(alpha)}", smooth_rbf)


def add_one_side_rbf(
    df: DataFrame, peak_at: int, alpha: float, date_col: str = "date"
) -> DataFrame:
    shrinkage = 52 / (peak_at * alpha**2)
    peak_rbf = F.exp(-shrinkage * (F.lit(peak_at) - F.weekofyear(date_col)) ** 2)

    return df.withColumn(f"one_side_rbf_{peak_at}_{int(alpha)}", peak_rbf)


def add_temporal_features(df: DataFrame) -> DataFrame:
    return (
        df
        # Periodic Exponential Peaks
        .transform(add_one_side_rbf, peak_at=52, alpha=10)
        .transform(add_one_side_rbf, peak_at=18, alpha=10)
        # Smooth week effects
        .transform(add_smooth_rbf, peak_at=52, alpha=15)
        .transform(add_smooth_rbf, peak_at=18, alpha=15)
        # Smooth month effects
        .transform(add_smooth_rbf, peak_at=50, alpha=75)
        .transform(add_smooth_rbf, peak_at=36, alpha=75)
    )

### Model Evaluation Funcs

In [ ]:
# def calculate_rmsle(df: DataFrame, label_col: str, pred_col: str) -> DataFrame:
#     return df.agg(
#         F.sqrt(F.mean(F.pow(F.log1p(label_col) - F.log1p(pred_col), 2)))
#     ).collect()[0][0]


# def score_model(
#     df: DataFrame, label_col: str, pred_col: str, baseline_pred_col: str, context: str
# ) -> dict[str, float]:
#     evaluator = RegressionEvaluator(labelCol=label_col, predictionCol=pred_col)
#     baseline_evaluator = RegressionEvaluator(
#         labelCol=label_col, predictionCol=baseline_pred_col
#     )

#     # Performance Metrics
#     mae = evaluator.evaluate(df, {evaluator.metricName: "mae"})
#     rmse = evaluator.evaluate(df, {evaluator.metricName: "rmse"})
#     rmsle = calculate_rmsle(df=df, label_col=label_col, pred_col=pred_col)

#     # Scaled Performance Metrics
#     mae_baseline = baseline_evaluator.evaluate(
#         df, {baseline_evaluator.metricName: "mae"}
#     )
#     rmse_baseline = baseline_evaluator.evaluate(
#         df, {baseline_evaluator.metricName: "rmse"}
#     )
#     rmsle_baseline = calculate_rmsle(
#         df=df, label_col=label_col, pred_col=baseline_pred_col
#     )

#     metrics = {
#         "rmsle": rmsle,
#         "rmse": rmse,
#         "mae": mae,
#         "scaled_rmsle": rmsle / rmsle_baseline,
#         "scaled_rmse": rmse / rmse_baseline,
#         "scaled_mae": mae / mae_baseline,
#     }
#     context_model_metrics = {f"{context}_{k}": v for k, v in metrics.items()}

#     return context_model_metrics


### Prototype Training

#### Feature Engineering

In [ ]:
train = df.filter(F.col("date") < cutoffs["train"])

default_agg_fns = [
    SparkAggFn("mean", F.mean),
    SparkAggFn("std", F.std),
    SparkAggFn("min", F.min),
    SparkAggFn("max", F.max),
    SparkAggFn("kurt", F.kurtosis),
    SparkAggFn("skew", F.skewness),
]

featured_train = (
    train.transform(
        add_features_per_grouping,
        grouping_cols=["store_id", "item_id"],
        lags=list(range(1, 5)),
        group_periods=[4, 8, 12],
        agg_spark_fns=default_agg_fns,
    )
    .transform(
        add_features_per_grouping,
        grouping_cols=["state", "city", "type", "cluster", "family", "class"],
        lags=list(range(1, 5)),
        group_periods=[4, 8, 12],
        agg_spark_fns=default_agg_fns,
    )
    .transform(
        add_features_per_grouping,
        grouping_cols=["family", "class"],
        lags=list(range(1, 5)),
        group_periods=[4, 8, 12],
        agg_spark_fns=default_agg_fns,
    )
    .transform(
        add_features_per_grouping,
        grouping_cols=["state", "city"],
        lags=list(range(1, 5)),
        group_periods=[4, 8, 12],
        agg_spark_fns=default_agg_fns,
    )
    .withColumn("perishable", F.col("perishable").cast("int").cast("float"))
)

#### Model Training

In [ ]:
# sample_fracs = (
#     calendar.filter(F.col("date") < cutoffs["train"])
#     .select(F.col("date").cast("string").alias("str_date"), F.lit(0.1).alias("frac"))
#     .toPandas()
#     .set_index("str_date")["frac"]
#     .to_dict()
# )

# sampled_featured_train = featured_train.withColumn(
#     "str_date", F.col("date").cast("string")
# ).sampleBy("str_date", fractions=sample_fracs)

# rows_per_date = (
#     sampled_featured_train.groupby("str_date").count().sort(F.asc("str_date"))
# )
# total_rows = sampled_featured_train.agg(F.count("*")).collect()[0][0]

# print(f"Total Rows = {total_rows}")
# rows_per_date.toPandas()

In [ ]:
exclude_cols = [
    "store_id",
    "date",
    "item_id",
    "time_percentage",
    "item_weight",
    "store_id_sale",
    "class",
    "first_product_store_purchase",
    "is_product_available",
    "unit_sale",
    "state_sale",
    "str_date",
    "state",
]
categorical_columns = [
    "family",
    "city",
    "cluster",
    "type",
    "perishable",
]
categorical_idx = [f"{cat_col}_idx" for cat_col in categorical_columns]

numeric_columns = [
    c
    for c in featured_train.columns
    if (c not in exclude_cols) and (c not in categorical_columns)
]

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import pandas as pd


num_features_corr = featured_train.select(*numeric_columns, "unit_sale").dropna()
assembler = VectorAssembler(inputCols=num_features_corr.columns, outputCol="features")
num_features_corr_vector = assembler.transform(num_features_corr)

corr_matrix = Correlation.corr(num_features_corr_vector, "features", method="spearman")
corr_res = pd.DataFrame(
    corr_matrix.collect()[0][0].toArray(),
    index=numeric_columns + ["unit_sale"],
    columns=numeric_columns + ["unit_sale"],
)

In [ ]:
xgb_preprocessing_pipe = Pipeline(
    stages=[
        StringIndexer(inputCols=categorical_columns, outputCols=categorical_idx),
        VectorAssembler(
            inputCols=numeric_columns + categorical_idx,
            outputCol="features",
            handleInvalid="skip",
        ),
    ]
)

lr_prep_pipe = Pipeline(
    stages=[
        VectorAssembler(
            inputCols=numeric_columns, outputCol="features", handleInvalid="skip"
        )
    ]
)

model_pipe = Pipeline(
    stages=[
        lr_prep_pipe,
        LinearRegression(labelCol="unit_sale")
        # # Specialized Model
        # SparkXGBRegressor(
        #     label_col="unit_sale",
        #     features_col="features",
        #     objective="reg:squarederror",
        #     tree_method="hist",
        #     grow_policy="lossguide",
        #     n_estimators=100,
        #     max_depth=8,
        #     subsample=0.9,
        #     num_workers=8,
        #     seed=0,
        # ),
    ]
)

reg_evaluator = RegressionEvaluator(labelCol="unit_sale", predictionCol="prediction")

model = model_pipe.fit(featured_train)
train_preds = model.transform(featured_train)

In [ ]:
score_model(
    train_preds.dropna(),
    baseline_pred_col="lag_unit_sale_1",
    context="train",
    label_col="unit_sale",
    pred_col="prediction",
)

In [ ]:
# # {"weight", "total_gain", "total_cover", "gain", "cover"}

# feat_index_map = pd.DataFrame(
#     data=[
#         (feat_idx, feature)
#         for feat_idx, feature in enumerate(numeric_columns + categorical_columns)
#     ],
#     columns=["feature_idx", "feature"],
# )

# _xgb_feat_importance = pd.DataFrame(
#     data=[
#         (int(feat_idx.replace("f", "")), feat_imp)
#         for feat_idx, feat_imp in model.stages[-1]
#         .get_feature_importances("gain")
#         .items()
#     ],
#     columns=["feature_idx", "feature_importance"],
# )

# feature_importance = (
#     pd.merge(feat_index_map, _xgb_feat_importance, how="left", on="feature_idx")
#     .sort_values(by="feature_importance", ascending=False)
#     .drop(columns=["feature_idx"])
# )

# feature_importance